In [1]:
import pandas as pd
import numpy as np

In [2]:
import time
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from sklearn.preprocessing import LabelEncoder
pd.set_option('display.width', 1000)

In [14]:
df = pd.read_csv('/Users/stephenkeyen/NLP projects/Hotel reviews/Hotel_Reviews_Filtered.csv')


In [4]:
import sys

if '/Users/stephenkeyen/Downloads/Functions 2' not in sys.path:
  sys.path.append('/Users/stephenkeyen/Downloads/Functions 2')

In [5]:
import function as fun

In [10]:
df_2 = pd.read_csv('/Users/stephenkeyen/NLP projects/Hotel reviews/Hotel_Summary.csv')

In [15]:
print(fun.metadata(df))

                                   column_name datatype  missing_percent  unique         mean          std        min          25%          50%          75%           max
0                                Hotel_Address      str             0.00       6          NaN          NaN        NaN          NaN          NaN          NaN           NaN
1                 Additional_Number_of_Scoring    int64             0.00     480   498.081836   500.538467   1.000000   169.000000   341.000000   660.000000   2682.000000
2                                  Review_Date      str             0.00     731          NaN          NaN        NaN          NaN          NaN          NaN           NaN
3                                Average_Score  float64             0.00      34     8.397487     0.548048   5.200000     8.100000     8.400000     8.800000      9.800000
4                                   Hotel_Name      str             0.00    1492          NaN          NaN        NaN          NaN          NaN  

In [11]:
print(fun.metadata(df_2))

               column_name datatype  missing_percent  unique         mean          std        min         25%         50%          75%           max
0               Hotel_Name      str              0.0    1475          NaN          NaN        NaN         NaN         NaN          NaN           NaN
1           Reviewer_Score  float64              0.0      28     8.313541     1.720605   2.500000    7.500000    8.800000     9.600000     10.000000
2  Total_Number_of_Reviews    int64              0.0    1132  1298.162492  1375.001969  43.000000  422.000000  867.000000  1679.000000  16670.000000
3                      lat  float64              0.0    1472    48.339336     3.415746  41.328376   45.533137   48.866856    51.504716     52.400181
4                      lng  float64              0.0    1472     3.966140     4.922368  -0.369758   -0.071492    2.307724     4.885964     16.429233
5                 Hotel_ID      str              0.0    1477          NaN          NaN        NaN         

In [16]:
df.dropna(inplace=True)

In [17]:
df.shape

(512470, 19)

In [18]:
display(df.groupby("Hotel_Address").agg({"Hotel_Name": "nunique"}))

,Hotel_Name
Hotel_Address,
"Amsterdam, Netherlands",105
"Barcelona, Spain",208
"London, United Kingdom",400
"Milan, Italy",162
"Paris, France",455
"Vienna, Austria",147


In [19]:
df['Hotel_ID'] = (
   df['Hotel_Name'].astype(str)
    + "_"
    + df['lat'].round(3).astype(str)
    + "_"
    + df['lng'].round(3).astype(str)
)

In [22]:
df['Total_Reviews_Found'] = (
    df.groupby('Hotel_ID')
    .Hotel_ID
    .transform('size') )

In [24]:


df['Average_Score'] = (
    df.groupby('Hotel_ID')['Reviewer_Score']
      .transform('mean')
      .round(1) )

In [26]:
# Drop `Additional_Number_of_Scoring`
df.drop(["Additional_Number_of_Scoring", "lat", "lng", "Total_Number_of_Reviews"], axis = 1, inplace=True)
# Replace `Total_Number_of_Reviews` and `Average_Score` with our own calculated values

## Tags processing

In [27]:
# Remove opening and closing brackets
df.Tags = df.Tags.str.strip("[']")
# remove all quotes too
df.Tags = df.Tags.str.replace(" ', '", ",", regex = False)

In [28]:
df.head(5)

,Hotel_Address,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,Calc_Average_Score,Average_Score_Difference,Hotel_ID,Total_Reviews_Found
0,"Amsterdam, Netherlands",8/3/2017,7.8,Hotel Arena,Russia,I am so angry that i made this post available...,397,Only the park outside of the hotel was beauti...,11,7,2.9,"Leisure trip, Couple, Duplex Double Room, Sta...",0 days,7.8,-0.1,Hotel Arena_52.361_4.916,405
1,"Amsterdam, Netherlands",8/3/2017,7.8,Hotel Arena,Ireland,No Negative,0,No real complaints the hotel was great great ...,105,7,7.5,"Leisure trip, Couple, Duplex Double Room, Sta...",0 days,7.8,-0.1,Hotel Arena_52.361_4.916,405
2,"Amsterdam, Netherlands",7/31/2017,7.8,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,Location was good and staff were ok It is cut...,21,9,7.1,"Leisure trip, Family with young children, Dup...",3 days,7.8,-0.1,Hotel Arena_52.361_4.916,405
3,"Amsterdam, Netherlands",7/31/2017,7.8,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,Great location in nice surroundings the bar a...,26,1,3.8,"Leisure trip, Solo traveler, Duplex Double Ro...",3 days,7.8,-0.1,Hotel Arena_52.361_4.916,405
4,"Amsterdam, Netherlands",7/24/2017,7.8,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,Amazing location and building Romantic setting,8,3,6.7,"Leisure trip, Couple, Suite, Stayed 2 nights,...",10 days,7.8,-0.1,Hotel Arena_52.361_4.916,405


### After reviewing the tag types and groups, the following was concluded

- df has 55k + unique tags, hard to create a standard version with that many tags, so we select popular universal tags important to understanding user's trip needs, who gave the review. 
- Ignoring tags showing user device type(irrelevant).
- Ignoring length of stay tags, it wouldn't be possible to define correlation between that and user stay satisfaction. 
- Ignoring tags with type of room due to different hotel definitions of room type which can't be standardized across hotel types. 
- We then settle to use only these tags that are consistent and with high volume and also help describe user's stay needs.

In [29]:
# Identified the most important tags as the following:
# Leisure trip, Couple, Solo traveler, Business trip, Group combined with Travelers with friends, 
# Family with young children, Family with older children, With a pet
df["Leisure_trip"] = df.Tags.apply(lambda tag: 1 if "Leisure trip" in tag else 0)
df["Couple"] = df.Tags.apply(lambda tag: 1 if "Couple" in tag else 0)
df["Solo_traveler"] = df.Tags.apply(lambda tag: 1 if "Solo traveler" in tag else 0)
df["Business_trip"] = df.Tags.apply(lambda tag: 1 if "Business trip" in tag else 0)
df["Group"] = df.Tags.apply(lambda tag: 1 if "Group" in tag or "Travelers with friends" in tag else 0)
df["Family_with_young_children"] = df.Tags.apply(lambda tag: 1 if "Family with young children" in tag else 0)
df["Family_with_older_children"] = df.Tags.apply(lambda tag: 1 if "Family with older children" in tag else 0)
df["With_a_pet"] = df.Tags.apply(lambda tag: 1 if "With a pet" in tag else 0)

In [30]:
# No longer need any of these columns
df.drop(["Review_Date", "Review_Total_Negative_Word_Counts", "Review_Total_Positive_Word_Counts", "days_since_review", "Total_Number_of_Reviews_Reviewer_Has_Given"], axis = 1, inplace=True)

In [32]:
print(fun.metadata(df))

                   column_name datatype  missing_percent  unique        mean         std  min    25%    50%     75%     max
0                Hotel_Address      str              0.0       6         NaN         NaN  NaN    NaN    NaN     NaN     NaN
1                Average_Score  float64              0.0      38    8.395813    0.610962  5.1    8.1    8.4     8.8     9.7
2                   Hotel_Name      str              0.0    1475         NaN         NaN  NaN    NaN    NaN     NaN     NaN
3         Reviewer_Nationality      str              0.0     227         NaN         NaN  NaN    NaN    NaN     NaN     NaN
4              Negative_Review      str              0.0  327927         NaN         NaN  NaN    NaN    NaN     NaN     NaN
5              Positive_Review      str              0.0  409941         NaN         NaN  NaN    NaN    NaN     NaN     NaN
6               Reviewer_Score  float64              0.0      37    8.395594    1.638170  2.5    7.5    8.8     9.6    10.0
7       

In [33]:
# No longer need any of these columns
df.drop(["Calc_Average_Score", "Average_Score_Difference", "Tags"], axis = 1, inplace=True)

In [34]:
# Saving new data file with calculated columns
print("Saving results to Hotel_Reviews_Final.csv")
df.to_csv(r'/Users/stephenkeyen/NLP projects/Hotel reviews/Hotel_Reviews_Final.csv', index = False)

Saving results to Hotel_Reviews_Final.csv
